In [39]:
import pandas as pd
import numpy as np
import duckdb
import pickle
MODELS_CACHE = {}

In [40]:
query = """
    SELECT
        pitcherId,
        Pitcher,
        pitchingTeam as Team,
        year,
        pitchType,
        pitchBucket,
        pitcherHand,
        batterHand,
        releaseVelocity,
        inducedVertBreak,
        horzBreak,
        spinRate,
        relX,
        relZ,
        extension,
        fb_velo,
        phand,
        bhand
    FROM
        "../../full_data.parquet"
    WHERE
        pitchType in ('FA', 'SI', 'FC', 'SL', 'CU', 'ST', 'CH', 'FS') AND
        year = 2026
"""
test = duckdb.sql(query).df()

In [41]:
def load_model(pitch):
    key = pitch
    if key not in MODELS_CACHE:
        with open(f'models/stuff_{pitch}.pkl', 'rb') as f:
            MODELS_CACHE[key] = pickle.load(f)
    return MODELS_CACHE[key]

In [42]:
fb_features = ["releaseVelocity", "inducedVertBreak", "horzBreak", "spinRate", "relX", "relZ", "extension", "phand", "bhand"]
non_fb_features = fb_features + ["fb_velo"]

test["xRV"] = np.nan

In [43]:
# One predict() call per pitchBucket
for pitch, group in test.groupby("pitchBucket"):
    features = fb_features if pitch in ("FB", "FC") else non_fb_features
    model = load_model(pitch)
    preds = model.predict(group[features])
    test.loc[group.index, "xRV"] = preds

In [44]:
test.sort_values("xRV")

,pitcherId,Pitcher,Team,year,pitchType,pitchBucket,pitcherHand,batterHand,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relX,relZ,extension,fb_velo,phand,bhand,xRV
521252,1182012928,C. Clark,CAMP,2026,FA,FB,R,L,95.68263,59.20960,11.79914,2336.692624,3.25016,7.65637,5.16032,90.810893,1,0,-0.016423
1448990,1213263616,D. Sheerin,LSU,2026,FA,FB,R,L,95.62755,17.42686,18.57273,2578.834687,3.74611,5.67022,5.82804,96.370309,1,0,-0.007788
182944,1341386496,N. Bonn,CP,2026,FA,FB,R,L,97.75589,22.23706,17.66001,2466.474559,2.47262,5.07375,5.92111,94.446143,1,0,-0.005679
1462308,1212299520,K. Kantola,LIP,2026,FA,FB,R,R,93.28416,21.44277,8.70189,2290.932390,1.64577,3.22335,7.01825,94.428167,1,1,-0.001620
1534640,1161348114,J. Music,CAMP,2026,FA,FB,R,L,93.35642,26.69812,13.79241,2336.325469,3.58698,5.78192,5.69298,91.241595,1,0,-0.000912
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248292,1342167684,C. Rodgers,LAF,2026,SI,FB,R,R,75.53439,-4.93075,6.16922,1852.110796,2.45673,3.42167,5.03552,76.419470,1,1,0.231419
1240772,1967526703,J. Davis,CIT,2026,FA,FB,R,R,76.10839,-5.61874,18.17359,1673.303260,2.52211,3.72924,3.97319,76.517811,1,1,0.231616
1240771,1967526703,J. Davis,CIT,2026,FA,FB,R,R,76.59708,-7.71835,18.90765,1727.345227,2.51016,3.83379,3.97416,76.517811,1,1,0.231779
1811709,1265732398,A. Bray,SMU,2026,FA,FB,L,R,76.71558,14.08568,-15.73410,1959.342515,-0.99520,5.48767,5.15438,78.325954,0,1,0.245334


In [45]:
stuff_split = test.groupby(["pitcherId", "pitchType", "batterHand"]).agg(
    Pitcher = ("Pitcher", "first"),
    team = ("Team", "first"),
    Hand = ("pitcherHand", "first"),
    Count = ("pitchType", "count"),
    releaseVelocity = ("releaseVelocity", "mean"),
    inducedVertBreak = ("inducedVertBreak", "mean"),
    horzBreak = ("horzBreak", "mean"),
    spinRate = ("spinRate", "mean"),
    relZ = ("relZ", "mean"),
    relX = ("relX", "mean"),
    extension = ("extension", "mean"),
    fb_velo = ("fb_velo", "mean"),
    xRV = ("xRV", "mean")
).reset_index()

In [46]:
stuff_split = stuff_split[stuff_split["Count"] >= 25]
stuff_vs_R = stuff_split[stuff_split["batterHand"] == "R"]
stuff_vs_L = stuff_split[stuff_split["batterHand"] == "L"]

In [47]:
stuff_split.sort_values("xRV").head(10)

,pitcherId,pitchType,batterHand,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV
8890,1152980224,SL,R,C. Stokes,FSU,R,117,85.191545,5.632745,-14.135156,2553.629680,5.512034,1.779805,5.499681,97.077668,0.031499
7089,1128102459,ST,R,D. Whitney,ORST,R,84,85.155016,-2.845933,-13.759070,2704.797876,6.374921,1.437509,5.842772,96.952494,0.033525
13817,1199169280,ST,R,A. Hutcheson,ORST,R,50,77.969551,7.460042,-20.452372,3076.285052,2.878659,3.574383,5.380931,88.538084,0.033833
15437,1213263616,ST,R,D. Sheerin,LSU,R,32,82.896181,0.484200,-16.372411,2772.956803,5.051798,3.708212,5.435280,96.370309,0.034038
26267,1330701056,SL,R,C. Benge,LSU,R,30,84.576302,6.861978,-12.993702,2489.102748,5.143121,2.033597,4.928346,95.473657,0.034411
37257,1595741236,ST,R,Z. Edwards,ORST,R,81,85.354137,6.105045,-11.221673,2306.727952,6.281953,1.193933,5.637858,95.851493,0.034432
7087,1128102459,SL,R,D. Whitney,ORST,R,91,86.653402,-0.692529,-12.027925,2632.582841,6.337235,1.545509,5.847296,96.952494,0.034495
37255,1595741236,SL,R,Z. Edwards,ORST,R,29,84.209863,5.372438,-10.181651,2192.492214,6.315088,1.600110,5.679649,95.851493,0.035974
6697,1123936875,SL,R,A. Buczkowski,CIN,R,208,81.351217,10.469901,-12.051962,2587.305479,3.652911,5.368686,5.314221,87.673389,0.037256
26871,1335999920,SL,R,B. Bryans,JVST,L,49,87.698346,2.245147,7.943053,2268.962475,4.531276,-4.714625,5.748146,94.838100,0.037806


In [48]:
stuff_split[(stuff_split["team"] == "URI")].sort_values("xRV").head(10)

,pitcherId,pitchType,batterHand,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV
33699,1487988943,SL,R,M. Santos,URI,R,150,81.681552,-0.511504,-16.697543,2654.874571,5.009123,2.670679,5.295196,92.562573,0.053916
6018,1110853376,SL,R,J. Sabbath,URI,R,206,78.702423,5.022416,-11.976576,2372.299814,5.674975,1.786428,5.846056,89.853506,0.055798
23002,1299756340,SL,L,P. Aikens,URI,L,79,72.952282,5.064905,16.641960,2660.951641,4.690285,-2.826688,4.240239,82.998339,0.059994
33698,1487988943,SL,L,M. Santos,URI,R,48,82.376851,0.136456,-15.434759,2634.539318,5.075633,2.606984,5.155787,92.562573,0.064299
23003,1299756340,SL,R,P. Aikens,URI,L,104,72.635064,5.439336,16.818101,2665.964923,4.675892,-2.817138,4.185233,82.998339,0.065820
8209,1143021056,SL,R,L. Lavigueur,URI,R,98,78.812354,-5.422396,-20.034032,2555.465549,5.907249,1.464444,5.291636,90.305237,0.068726
17822,1241946368,SL,R,D. Asencio,URI,R,28,74.535699,4.035159,-14.339310,2391.403672,5.853002,1.742757,5.549577,86.591519,0.068862
6017,1110853376,SL,L,J. Sabbath,URI,R,47,78.605516,3.878899,-10.361765,2345.508683,5.714029,1.836662,5.732389,89.853506,0.068950
6012,1110853376,CU,R,J. Sabbath,URI,R,63,75.704597,-4.018063,-12.546962,2365.669755,5.874202,1.439003,5.774049,89.853506,0.069466
8208,1143021056,SL,L,L. Lavigueur,URI,R,32,78.325602,-3.832544,-19.841544,2530.095337,5.860259,1.597343,5.276408,90.305237,0.070721


In [49]:
stuff_vs_R.sort_values("xRV").head(10)

,pitcherId,pitchType,batterHand,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV
8890,1152980224,SL,R,C. Stokes,FSU,R,117,85.191545,5.632745,-14.135156,2553.629680,5.512034,1.779805,5.499681,97.077668,0.031499
7089,1128102459,ST,R,D. Whitney,ORST,R,84,85.155016,-2.845933,-13.759070,2704.797876,6.374921,1.437509,5.842772,96.952494,0.033525
13817,1199169280,ST,R,A. Hutcheson,ORST,R,50,77.969551,7.460042,-20.452372,3076.285052,2.878659,3.574383,5.380931,88.538084,0.033833
15437,1213263616,ST,R,D. Sheerin,LSU,R,32,82.896181,0.484200,-16.372411,2772.956803,5.051798,3.708212,5.435280,96.370309,0.034038
26267,1330701056,SL,R,C. Benge,LSU,R,30,84.576302,6.861978,-12.993702,2489.102748,5.143121,2.033597,4.928346,95.473657,0.034411
37257,1595741236,ST,R,Z. Edwards,ORST,R,81,85.354137,6.105045,-11.221673,2306.727952,6.281953,1.193933,5.637858,95.851493,0.034432
7087,1128102459,SL,R,D. Whitney,ORST,R,91,86.653402,-0.692529,-12.027925,2632.582841,6.337235,1.545509,5.847296,96.952494,0.034495
37255,1595741236,SL,R,Z. Edwards,ORST,R,29,84.209863,5.372438,-10.181651,2192.492214,6.315088,1.600110,5.679649,95.851493,0.035974
6697,1123936875,SL,R,A. Buczkowski,CIN,R,208,81.351217,10.469901,-12.051962,2587.305479,3.652911,5.368686,5.314221,87.673389,0.037256
26871,1335999920,SL,R,B. Bryans,JVST,L,49,87.698346,2.245147,7.943053,2268.962475,4.531276,-4.714625,5.748146,94.838100,0.037806


In [50]:
stuff_vs_L.sort_values("xRV").head(10)

,pitcherId,pitchType,batterHand,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV
7080,1128102459,CU,L,D. Whitney,ORST,R,41,76.930017,-16.254943,-11.481498,2724.544600,6.491244,1.156354,5.940329,96.952494,0.040978
9461,1161348114,FA,L,J. Music,CAMP,R,105,92.444064,21.094367,13.419586,2220.296852,5.962211,3.536584,5.697732,91.241595,0.043479
30213,1362777296,SL,L,J. Bauer,MSST,L,47,84.822591,-4.775445,17.203654,2755.073655,5.766569,-2.288755,4.782764,97.037448,0.044000
8889,1152980224,SL,L,C. Stokes,FSU,R,50,84.399287,4.484846,-15.166882,2545.817537,5.514327,1.821089,5.444994,97.077668,0.044326
28436,1342168139,CU,L,C. Jasa,NEB,R,98,79.541574,-15.412199,-12.641105,2826.470878,6.550808,1.032867,5.368775,95.056906,0.044415
47524,1917143765,CU,L,T. Thames,RICE,R,79,81.546504,-14.143327,-6.581540,2161.735593,6.572734,1.417166,5.081821,94.943000,0.045565
24824,1310973952,SL,L,S. Sdao,TXAM,L,90,80.943233,7.024978,10.862263,2543.279130,5.408778,-3.099979,5.970216,91.355027,0.045953
49580,1985708358,CU,L,E. Lund,OKST,L,96,83.032349,-10.176805,2.958382,2624.600285,6.384888,-1.626147,6.058088,93.499447,0.046143
23119,1301090561,CU,L,H. Dietz,ARK,L,25,80.572646,-14.080084,7.455636,2718.451529,6.683361,-2.096690,5.319958,91.365488,0.046156
25510,1319280896,CU,L,M. Yehl,WVU,L,81,83.803672,-8.404339,12.796511,2569.046387,6.045593,-3.025660,5.651696,91.794644,0.046195


In [51]:
from scipy import stats

stuff_vs_R["Stuff+"] = 100 + (-10 * stuff_vs_R.groupby("pitchType")["xRV"].transform(stats.zscore))
stuff_vs_R["Stuff+"] = stuff_vs_R["Stuff+"].round().astype(int)

stuff_vs_L["Stuff+"] = 100 + (-10 * stuff_vs_L.groupby("pitchType")["xRV"].transform(stats.zscore))
stuff_vs_L["Stuff+"] = stuff_vs_L["Stuff+"].round().astype(int)

/var/folders/lr/9t8mdvqx6_58bf8fjvwdbh3h0000gp/T/ipykernel_70488/220064244.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stuff_vs_R["Stuff+"] = 100 + (-10 * stuff_vs_R.groupby("pitchType")["xRV"].transform(stats.zscore))
/var/folders/lr/9t8mdvqx6_58bf8fjvwdbh3h0000gp/T/ipykernel_70488/220064244.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stuff_vs_R["Stuff+"] = stuff_vs_R["Stuff+"].round().astype(int)
/var/folders/lr/9t8mdvqx6_58bf8fjvwdbh3h0000gp/T/ipykernel_70488/220064244.py:6: SettingWithCo

In [52]:
print("R: ", len(stuff_vs_R))
print("L: ", len(stuff_vs_L))

R:  11308
L:  8422


In [53]:
stuff_vs_R.sort_values("Stuff+", ascending=False)

,pitcherId,pitchType,batterHand,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
44750,1823013818,FC,R,B. Mannis,AC,L,85,82.765076,0.347540,3.365631,2498.557528,6.589466,-0.985725,5.545165,86.737167,0.059986,139
49434,1979617275,FC,R,D. Volantis,TEX,L,211,86.492859,0.418523,-1.725350,2542.786322,6.736020,-2.283476,5.417244,89.767069,0.062173,135
40781,1698188238,FC,R,W. Jordan,TTU,R,36,80.122987,0.088541,-0.676375,2239.583607,6.743496,1.655561,5.169522,86.088303,0.062513,134
34694,1518451927,FC,R,A. Berggren,M-OH,R,98,89.071619,-1.249334,-5.286562,2954.159864,5.816047,1.291455,5.243581,90.948463,0.062665,134
11615,1181538305,FC,R,H. Hamilton,TEX,R,74,88.682779,6.874224,-4.905706,2648.046095,6.335441,0.914348,5.015254,90.815047,0.063179,133
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1281,1040181757,SL,R,E. Lichtenauer,OHIO,L,66,70.148711,3.358026,0.041426,1784.532074,5.555643,-1.155297,4.645945,78.773269,0.124670,64
7519,1133497856,SL,R,H. Konkler,ALCN,R,58,68.439069,-6.293904,-13.665053,2208.261770,5.773662,1.614555,6.198790,78.209598,0.126033,63
8719,1150052244,SL,R,J. Burt,MRST,R,38,71.325699,0.341763,-8.774383,1823.958036,5.042263,1.588116,6.684595,83.447137,0.127810,62
1271,1040181757,CH,R,E. Lichtenauer,OHIO,L,33,75.481570,12.750966,-10.637911,1712.087116,5.548175,-0.896256,4.825652,78.773269,0.148896,61


In [54]:
stuff_vs_L.sort_values("Stuff+", ascending=False)

,pitcherId,pitchType,batterHand,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
9461,1161348114,FA,L,J. Music,CAMP,R,105,92.444064,21.094367,13.419586,2220.296852,5.962211,3.536584,5.697732,91.241595,0.043479,136
44749,1823013818,FC,L,B. Mannis,AC,L,64,82.976581,1.096988,2.489352,2470.052884,6.572577,-0.796770,5.601070,86.737167,0.062617,134
35943,1557710177,FA,L,J. Nottingham,UGA,R,70,96.101416,20.866880,12.678327,2589.355888,6.179710,1.600505,5.933112,95.237863,0.047881,133
49433,1979617275,FC,L,D. Volantis,TEX,L,57,86.908055,0.398738,-0.809702,2557.026393,6.763803,-2.114986,5.470278,89.767069,0.063590,132
17176,1237823232,SI,L,R. Gannon,ILL,L,149,87.501590,0.532602,-20.168874,1969.176531,4.805830,-2.947499,5.717237,87.096623,0.061048,132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1280,1040181757,SL,L,E. Lichtenauer,OHIO,L,45,69.740735,-0.700655,4.115441,1846.471997,5.556583,-1.088761,4.597890,78.773269,0.128368,56
5309,1100871680,FC,L,C. Fennell,VAN,R,25,81.987073,9.256024,1.445839,1880.015340,4.601299,2.372368,6.989905,87.354242,0.106679,55
1272,1040181757,CU,L,E. Lichtenauer,OHIO,L,34,68.865538,-2.181682,4.993084,1909.532286,5.439110,-1.141353,4.346725,78.773269,0.131892,53
17133,1237204199,SL,L,D. Jackson,CSB,L,52,70.022381,0.866272,0.282074,1916.642137,5.422629,-2.982960,5.112957,80.242872,0.133210,52


In [55]:
stuff_vs_R[(stuff_vs_R["team"] == "URI")].sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,batterHand,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
33696,1487988943,FA,R,M. Santos,URI,R,133,92.617313,16.315802,7.248097,2482.995831,5.286362,2.446696,5.642211,92.562573,0.075957,116
30005,1358089441,FA,R,J. Cullen,URI,R,207,91.655132,21.389731,10.161991,2264.011820,6.098269,1.076454,6.028869,89.111456,0.075146,116
14353,1203468032,FA,R,C. Grotyohann,URI,R,119,90.718167,21.181017,8.681576,2194.065073,6.185525,1.837099,5.884816,88.676504,0.076845,116
33699,1487988943,SL,R,M. Santos,URI,R,150,81.681552,-0.511504,-16.697543,2654.874571,5.009123,2.670679,5.295196,92.562573,0.053916,115
6018,1110853376,SL,R,J. Sabbath,URI,R,206,78.702423,5.022416,-11.976576,2372.299814,5.674975,1.786428,5.846056,89.853506,0.055798,113
26915,1336247552,FA,R,C. Maloney,URI,R,52,90.717560,18.498317,9.614368,2206.545021,5.818546,2.185176,5.979694,90.394947,0.085065,111
6014,1110853376,FA,R,J. Sabbath,URI,R,260,89.868966,18.524338,2.068743,2240.852106,5.804849,1.526865,6.486560,89.853506,0.086035,111
30007,1358089441,FC,R,J. Cullen,URI,R,106,82.532377,2.165631,-4.745621,2600.376580,5.836396,1.643331,5.508675,89.111456,0.074420,110
3376,1085415168,CH,R,E. Maloney,URI,R,72,75.901056,9.947582,9.387866,1463.733821,6.954468,0.010373,4.892221,87.229034,0.075225,110
5347,1101042688,FA,R,A. Jones,URI,R,109,89.738485,21.030548,7.680301,2157.222519,5.831305,1.049692,5.835262,88.680534,0.088338,109


In [56]:
stuff_vs_L[(stuff_vs_L["team"] == "URI")].sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,batterHand,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
30004,1358089441,FA,L,J. Cullen,URI,R,134,91.504425,21.558123,10.812954,2247.488274,6.109855,1.158227,5.966145,89.111456,0.071368,120
14352,1203468032,FA,L,C. Grotyohann,URI,R,54,90.773486,20.555848,9.609507,2169.743218,6.191051,1.965801,5.876944,88.676504,0.081711,115
23002,1299756340,SL,L,P. Aikens,URI,L,79,72.952282,5.064905,16.641960,2660.951641,4.690285,-2.826688,4.240239,82.998339,0.059994,114
3375,1085415168,CH,L,E. Maloney,URI,R,42,75.415503,10.167633,9.908102,1430.106327,6.900247,0.168996,4.891595,87.229034,0.073735,113
26914,1336247552,FA,L,C. Maloney,URI,R,44,90.575821,19.305781,9.735882,2205.921560,5.811788,2.239379,5.948470,90.394947,0.089395,111
33698,1487988943,SL,L,M. Santos,URI,R,48,82.376851,0.136456,-15.434759,2634.539318,5.075633,2.606984,5.155787,92.562573,0.064299,110
5346,1101042688,FA,L,A. Jones,URI,R,34,89.504939,21.096476,8.030556,2155.581368,5.847140,1.068843,5.833574,88.680534,0.095779,107
12882,1194448803,FA,L,S. Houchens,URI,L,86,88.225763,19.425271,-10.727676,2133.799044,6.238912,-1.716146,5.494089,87.959847,0.097493,106
6017,1110853376,SL,L,J. Sabbath,URI,R,47,78.605516,3.878899,-10.361765,2345.508683,5.714029,1.836662,5.732389,89.853506,0.068950,106
8208,1143021056,SL,L,L. Lavigueur,URI,R,32,78.325602,-3.832544,-19.841544,2530.095337,5.860259,1.597343,5.276408,90.305237,0.070721,105


In [57]:
stuff = test.groupby(["pitcherId", "pitchType"]).agg(
    Pitcher = ("Pitcher", "first"),
    team = ("Team", "first"),
    Hand = ("pitcherHand", "first"),
    Count = ("pitchType", "count"),
    releaseVelocity = ("releaseVelocity", "mean"),
    inducedVertBreak = ("inducedVertBreak", "mean"),
    horzBreak = ("horzBreak", "mean"),
    spinRate = ("spinRate", "mean"),
    relZ = ("relZ", "mean"),
    relX = ("relX", "mean"),
    extension = ("extension", "mean"),
    fb_velo = ("fb_velo", "mean"),
    xRV = ("xRV", "mean")
).reset_index()

In [58]:
stuff = stuff[stuff["Count"] >= 25]

In [59]:
stuff.sort_values("xRV").head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV
8541,1213263616,ST,D. Sheerin,LSU,R,35,82.901609,0.417314,-16.230077,2770.899364,5.046791,3.704743,5.457363,96.370309,0.034932
14527,1330701056,SL,C. Benge,LSU,R,32,84.622070,6.698221,-13.297860,2490.842436,5.142583,2.042865,4.934933,95.473657,0.034965
3925,1128102459,ST,D. Whitney,ORST,R,103,85.102919,-2.945456,-13.999854,2707.779337,6.371775,1.448231,5.844042,96.952494,0.035210
4927,1152980224,SL,C. Stokes,FSU,R,167,84.954342,5.289063,-14.444056,2551.290715,5.512721,1.792165,5.483308,97.077668,0.035340
3924,1128102459,SL,D. Whitney,ORST,R,114,86.633362,-0.488421,-11.598474,2618.534271,6.340307,1.559128,5.848839,96.952494,0.036980
20655,1595741236,ST,Z. Edwards,ORST,R,125,85.435751,6.560465,-10.999986,2298.276092,6.268728,1.204577,5.662743,95.851493,0.039001
7652,1199169280,ST,A. Hutcheson,ORST,R,71,78.045718,7.586663,-20.358406,3056.355302,2.887788,3.448694,5.356386,88.538084,0.039429
20654,1595741236,SL,Z. Edwards,ORST,R,50,84.320288,5.197996,-9.977271,2221.019658,6.292231,1.553459,5.679015,95.851493,0.041254
9848,1241464169,SL,C. Markham,ORE,R,35,86.507693,5.638884,-10.116782,2383.240312,6.009514,1.361420,5.775899,94.654776,0.041496
27509,1984268337,CU,J. Robertson,MISS,R,59,83.823267,4.570024,-13.940972,2208.393349,5.652838,2.872557,5.770272,95.982464,0.041743


In [60]:
stuff[(stuff["team"] == "URI")].sort_values("xRV").head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV
18671,1487988943,SL,M. Santos,URI,R,198,81.850110,-0.354423,-16.391414,2649.944813,5.025247,2.655238,5.261400,92.562573,0.056433
3325,1110853376,SL,J. Sabbath,URI,R,253,78.684421,4.809984,-11.676591,2367.322806,5.682230,1.795760,5.824940,89.853506,0.058242
12730,1299756340,SL,P. Aikens,URI,L,183,72.772005,5.277696,16.742062,2663.800719,4.682106,-2.821260,4.208979,82.998339,0.063305
4557,1143021056,SL,L. Lavigueur,URI,R,130,78.692538,-5.031048,-19.986650,2549.220573,5.895682,1.497158,5.287887,90.305237,0.069217
16613,1358089441,CU,J. Cullen,URI,R,63,76.407241,-13.357738,-14.064611,2764.255015,6.194429,1.217194,5.257873,89.111456,0.070125
3322,1110853376,CU,J. Sabbath,URI,R,78,75.363774,-4.658012,-12.280351,2362.144250,5.881616,1.443322,5.763144,89.853506,0.070931
7138,1194448803,SL,S. Houchens,URI,L,267,79.853445,2.077754,2.822317,2309.006842,5.732522,-2.283857,5.391965,87.959847,0.073038
2954,1101042688,CU,A. Jones,URI,R,40,76.407526,-17.789101,-10.938031,2581.275308,5.843453,1.020377,5.316856,88.680534,0.073143
9867,1241946368,SL,D. Asencio,URI,R,37,74.932389,4.582871,-12.024322,2344.111408,5.844129,1.746269,5.567245,86.591519,0.073376
16614,1358089441,FA,J. Cullen,URI,R,341,91.595910,21.455902,10.417795,2257.518696,6.102822,1.108587,6.004221,89.111456,0.073661


In [61]:
stuff["Stuff+"] = 100 + (-10 * stuff.groupby("pitchType")["xRV"].transform(stats.zscore))
stuff["Stuff+"] = stuff["Stuff+"].round().astype(int)

In [62]:
stuff.sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
24840,1823013818,FC,B. Mannis,AC,L,149,82.855924,0.669450,2.989243,2486.313922,6.582212,-0.904563,5.569178,86.737167,0.061116,137
24718,1817575025,FC,A. Yearwood,TXST,R,26,84.732237,5.608424,-0.765390,2299.229962,6.785858,0.380747,5.674408,89.172526,0.061853,136
11996,1285478656,FC,D. Pressley,WEBB,R,32,82.712190,6.571443,0.581246,2287.814126,7.018037,0.882721,5.080079,86.430384,0.061989,136
7812,1201847438,SI,K. Hawks,MORE,L,32,89.435034,20.402186,-15.960743,2149.084340,6.309775,-2.123682,5.866350,88.703101,0.067144,135
27446,1979617275,FC,D. Volantis,TEX,L,268,86.581166,0.414315,-1.530604,2545.814993,6.741929,-2.247640,5.428524,89.767069,0.062474,135
18367,1466686191,SI,S. Moore,MICH,L,49,90.776254,12.947649,-17.231033,2234.948992,4.950867,-3.701829,5.943492,90.946537,0.068475,134
20869,1602557566,FA,S. Garcia,LSU,L,157,92.719318,21.283184,-10.994845,2441.646707,6.250958,-1.623876,6.137295,92.141575,0.049330,133
5243,1161348114,FA,J. Music,CAMP,R,212,92.098181,21.348214,13.037600,2209.426986,5.969218,3.512801,5.712665,91.241595,0.049431,132
6437,1181538305,FC,H. Hamilton,TEX,R,107,88.841814,7.059333,-4.996975,2653.573051,6.328644,0.922440,5.021873,90.815047,0.064053,132
7485,1197348608,FA,R. Marohn,NCST,L,381,91.792761,17.194621,-13.966061,2262.851961,5.568628,-3.642764,6.151826,91.768831,0.051847,131


In [63]:
stuff[stuff["pitchType"] == "FA"].sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
20869,1602557566,FA,S. Garcia,LSU,L,157,92.719318,21.283184,-10.994845,2441.646707,6.250958,-1.623876,6.137295,92.141575,0.049330,133
5243,1161348114,FA,J. Music,CAMP,R,212,92.098181,21.348214,13.037600,2209.426986,5.969218,3.512801,5.712665,91.241595,0.049431,132
25541,1862463014,FA,J. Barberi,FLA,R,305,97.274069,21.009392,11.024492,2534.407232,6.332058,1.838098,5.404268,97.170550,0.051750,131
19919,1557710177,FA,J. Nottingham,UGA,R,157,95.982653,21.027460,12.168996,2557.922517,6.179032,1.538617,5.987786,95.237863,0.052965,131
1595,1082403072,FA,S. Fitzpatrick,ASU,L,153,91.981386,14.356447,-15.377900,2283.033920,4.650294,-3.252079,6.171938,91.973832,0.051904,131
7485,1197348608,FA,R. Marohn,NCST,L,381,91.792761,17.194621,-13.966061,2262.851961,5.568628,-3.642764,6.151826,91.768831,0.051847,131
18305,1464054215,FA,S. Sandford,FLA,R,233,95.154367,17.736087,10.730547,2297.266033,5.905669,3.306439,6.304571,95.088761,0.054723,130
27088,1956935142,FA,C. Clark,USM,R,488,93.180759,21.361006,9.224956,2263.027148,5.491940,1.819227,6.491275,93.104134,0.053077,130
15507,1342167297,FA,P. Manca,FSU,L,110,90.874663,22.622299,-9.108392,2406.470879,6.699013,-1.694085,5.833289,90.874663,0.053041,130
3474,1116162816,FA,C. Howard,TEX,R,118,94.293026,16.267949,8.834254,2155.550396,5.001592,3.648234,6.572414,92.901081,0.056623,129


In [64]:
stuff[stuff["pitchType"] == "SI"].sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
7812,1201847438,SI,K. Hawks,MORE,L,32,89.435034,20.402186,-15.960743,2149.084340,6.309775,-2.123682,5.866350,88.703101,0.067144,135
18367,1466686191,SI,S. Moore,MICH,L,49,90.776254,12.947649,-17.231033,2234.948992,4.950867,-3.701829,5.943492,90.946537,0.068475,134
21197,1621642851,SI,U. Fernsler,TCU,L,70,90.769362,11.962531,-16.986189,2344.955094,4.874789,-3.283593,6.705121,90.344340,0.072732,130
11354,1275096832,SI,K. Johnson,UVA,L,27,93.358368,13.376516,-16.324775,2318.129342,5.624131,-3.112297,6.097339,92.714601,0.072720,130
7536,1198395392,SI,S. Matson,USC,L,43,90.231300,13.005725,-17.409117,2277.046311,4.820987,-1.716367,6.337552,90.121369,0.076594,127
26663,1933129624,SI,A. Weiss,MD,L,40,92.386510,11.464331,-18.365267,2042.024588,5.218378,-3.069507,5.692496,91.984655,0.077987,126
1419,1077565952,SI,E. Smith,MD,L,99,90.371054,14.648734,-19.879479,2307.082267,5.211704,-1.713970,5.750767,90.269006,0.077795,126
171,1008376863,SI,D. Hamilton,CARK,R,62,90.468106,17.620931,17.544432,2455.711160,5.122982,2.492114,5.622249,90.302159,0.078085,126
14849,1335271936,SI,E. Norby,ECU,L,42,91.305717,11.962082,-13.278088,2551.436736,5.133850,-2.885767,6.357592,90.769815,0.077804,126
26634,1930666388,SI,E. Plog,LSU,L,105,93.319909,7.144408,-18.442580,2430.241444,5.346407,-2.389048,5.540810,93.241537,0.079548,125


In [65]:
stuff[(stuff["pitchType"] == "CH")].sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
14887,1336293657,CH,B. Reiter,PITT,L,83,78.460375,6.617510,-22.098956,2268.411656,5.162403,-1.178213,5.929052,91.477784,0.049690,130
7710,1200115665,CH,L. Hood,GONZ,R,229,77.619751,8.521231,18.575005,2097.610552,5.275350,1.847231,7.021906,93.520070,0.050494,129
12460,1296842752,CH,G. Naess,CP,R,343,75.937300,7.601743,19.442501,2086.549300,6.454047,1.545548,5.893917,87.430316,0.053732,127
11636,1279321088,CH,C. Carlon,ASU,L,45,83.611771,6.382310,-11.167124,1130.573376,6.510748,-0.991031,5.620100,95.532941,0.055131,126
20867,1602557566,CH,S. Garcia,LSU,L,32,80.533450,11.860993,-16.906325,2330.071110,5.870718,-2.158693,6.250992,92.141575,0.056289,125
2902,1100788992,CH,A. Weaver,DUKE,R,326,80.066990,10.742349,12.755850,1704.081592,6.394834,1.704192,6.934339,93.935864,0.055952,125
4114,1132480961,CH,P. Warner,STAN,R,192,81.147656,3.283221,21.815042,2151.563225,5.628583,2.357551,5.644811,91.307842,0.056449,125
13198,1307080960,CH,B. Purcell,FSU,R,109,85.104335,4.429942,15.105850,1334.006154,6.741752,0.370746,5.073430,91.956377,0.055584,125
21208,1621942203,CH,T. Tracey,TENN,L,83,78.845313,11.442121,-17.369830,2057.680349,5.777248,-1.501100,5.690838,90.776792,0.056956,125
26004,1893153714,CH,T. Martin,ME,R,37,75.873275,9.716764,16.496335,2024.969755,6.329656,0.509220,5.835893,88.469405,0.057759,124


In [66]:
stuff[stuff["pitchType"] == "FS"].sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
9797,1239346432,FS,H. Leffew,TEX,L,27,82.912730,1.497326,-3.315634,1575.579788,6.767964,-0.660123,5.585791,92.279265,0.053438,127
4513,1142711296,FS,J. Noot,LSU,R,44,87.712332,-0.154011,16.836437,1513.930355,6.158896,1.905507,4.926131,92.904013,0.057616,123
13012,1305389198,FS,B. Shannon,LOU,R,107,90.325924,1.835266,13.528682,1386.975724,5.990299,2.407847,5.296464,95.962592,0.060844,121
2861,1100603904,FS,T. Beard,FSU,L,43,83.618835,4.580568,-4.144414,1438.631334,6.535243,-0.700984,5.424319,91.142982,0.060808,121
3494,1116314624,FS,A. Solis,HOU,R,30,83.252192,6.172707,13.603791,1250.166372,6.127030,0.718687,5.125896,94.141402,0.060941,120
5250,1161417432,FS,C. Gomez,STAN,R,26,82.946165,9.306071,7.200483,1088.876713,6.822493,1.064298,5.921258,91.534534,0.063859,118
13001,1305069568,FS,S. Hall,TXST,R,80,83.611406,4.353146,11.578021,1133.095415,6.456191,0.249012,4.990553,90.819006,0.064461,118
3596,1120804043,FS,L. Kenny,UOWG,R,74,82.049659,-0.034706,13.399857,997.282707,6.282034,1.035032,5.329757,90.683435,0.065622,117
445,1024294920,FS,B. Curry,TOL,R,81,81.198953,7.389476,7.219858,1895.150431,6.556151,0.893582,5.724129,92.561824,0.064862,117
4896,1152932096,FS,W. Wiatrek,UTRGV,R,75,79.492451,8.165667,13.891930,1347.580131,6.790957,0.558930,6.037865,87.831669,0.068497,114


In [70]:
stuff[(stuff["pitchType"] == "SL")].sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
14527,1330701056,SL,C. Benge,LSU,R,32,84.622070,6.698221,-13.297860,2490.842436,5.142583,2.042865,4.934933,95.473657,0.034965,130
4927,1152980224,SL,C. Stokes,FSU,R,167,84.954342,5.289063,-14.444056,2551.290715,5.512721,1.792165,5.483308,97.077668,0.035340,130
3924,1128102459,SL,D. Whitney,ORST,R,114,86.633362,-0.488421,-11.598474,2618.534271,6.340307,1.559128,5.848839,96.952494,0.036980,129
9848,1241464169,SL,C. Markham,ORE,R,35,86.507693,5.638884,-10.116782,2383.240312,6.009514,1.361420,5.775899,94.654776,0.041496,126
20654,1595741236,SL,Z. Edwards,ORST,R,50,84.320288,5.197996,-9.977271,2221.019658,6.292231,1.553459,5.679015,95.851493,0.041254,126
16734,1362777296,SL,J. Bauer,MSST,L,77,84.642956,-4.855708,17.179466,2720.335755,5.763301,-2.308990,4.758917,97.037448,0.041876,125
25543,1862463014,SL,J. Barberi,FLA,R,259,85.306334,-0.671758,-7.204715,2635.780381,6.121268,2.006186,5.061015,97.170550,0.043134,124
18155,1457514761,SL,L. Eldem,FGCU,R,44,83.526161,0.215397,-14.046762,2456.803850,6.093375,1.732476,5.987577,94.337342,0.044234,124
10315,1254299136,SL,J. Flora,UCSB,R,246,84.904103,5.696826,-11.408453,2616.862456,5.573449,2.375820,5.851724,95.280077,0.043228,124
3704,1123936875,SL,A. Buczkowski,CIN,R,340,81.096458,10.655022,-12.481602,2580.724594,3.641327,5.368342,5.300690,87.673389,0.043369,124


In [71]:
stuff[(stuff["pitchType"] == "ST")].sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
8541,1213263616,ST,D. Sheerin,LSU,R,35,82.901609,0.417314,-16.230077,2770.899364,5.046791,3.704743,5.457363,96.370309,0.034932,128
3925,1128102459,ST,D. Whitney,ORST,R,103,85.102919,-2.945456,-13.999854,2707.779337,6.371775,1.448231,5.844042,96.952494,0.035210,127
7652,1199169280,ST,A. Hutcheson,ORST,R,71,78.045718,7.586663,-20.358406,3056.355302,2.887788,3.448694,5.356386,88.538084,0.039429,124
20655,1595741236,ST,Z. Edwards,ORST,R,125,85.435751,6.560465,-10.999986,2298.276092,6.268728,1.204577,5.662743,95.851493,0.039001,124
10316,1254299136,ST,J. Flora,UCSB,R,219,80.965482,1.258499,-17.519195,2679.096799,5.444058,2.448901,5.825916,95.280077,0.044467,120
19931,1558099259,ST,K. Sweum,GONZ,L,65,83.703538,-2.452340,11.641476,2579.655366,6.219456,-1.850143,5.477431,91.602552,0.048205,117
5903,1166948690,ST,N. Moore,TXST,L,30,81.690203,-0.498679,10.302095,2869.817993,6.357798,-1.430169,4.677549,93.597661,0.049175,116
13742,1311020288,ST,J. Volchko,UGA,R,38,87.479000,2.600799,-12.837503,2859.625805,5.484037,2.401896,5.826063,94.463887,0.048644,116
22790,1709079320,ST,E. Kleinschmit,ORST,L,110,79.202612,-1.489185,19.036217,2747.411336,5.986380,-1.780200,5.546254,91.326881,0.049729,115
8319,1208166656,ST,L. Harrison,TEX,L,44,81.306242,5.691617,12.432429,2534.064615,5.307758,-3.510227,5.596603,90.538881,0.050135,115


In [72]:
stuff[(stuff["pitchType"] == "CU")].sort_values("Stuff+", ascending=False).head(10)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
27509,1984268337,CU,J. Robertson,MISS,R,59,83.823267,4.570024,-13.940972,2208.393349,5.652838,2.872557,5.770272,95.982464,0.041743,125
27526,1985708358,CU,E. Lund,OKST,L,397,82.886103,-10.720096,3.019373,2637.242411,6.383948,-1.657159,6.017577,93.499447,0.042202,125
3920,1128102459,CU,D. Whitney,ORST,R,66,77.272689,-16.352383,-11.240323,2728.911252,6.498169,1.146138,5.954857,96.952494,0.041869,125
12795,1301090561,CU,H. Dietz,ARK,L,137,80.187390,-14.573246,7.765929,2743.260361,6.724518,-2.184754,5.278326,91.365488,0.043466,124
20348,1580729364,CU,T. Valincius,MSST,L,50,84.228467,-4.624515,12.461339,2808.986724,5.842505,-2.024922,5.693494,94.537579,0.044367,123
14107,1319280896,CU,M. Yehl,WVU,L,294,83.840434,-9.210736,12.518104,2592.973728,6.014400,-3.031657,5.649792,91.794644,0.044548,123
15727,1342168139,CU,C. Jasa,NEB,R,196,79.780793,-15.477776,-12.517007,2828.139631,6.564018,1.003084,5.392045,95.056906,0.045998,122
23536,1752397689,CU,H. Vincent,TXAM,L,95,82.432109,-11.545562,7.348955,2423.087010,6.155636,-2.885196,5.217131,92.771624,0.045661,122
8067,1205369204,CU,N. Yoder,UVA,R,115,85.285663,-11.789671,-5.792911,2585.738258,6.154267,1.839658,5.986802,95.877499,0.046607,122
13149,1306914798,CU,R. Bowie,WAKE,L,99,81.418479,-15.038355,13.518751,2743.339634,5.159636,-1.864873,5.431981,94.456925,0.046181,122


In [73]:
stuff[stuff["team"] == "URI"].sort_values("Stuff+", ascending=False)

,pitcherId,pitchType,Pitcher,team,Hand,Count,releaseVelocity,inducedVertBreak,horzBreak,spinRate,relZ,relX,extension,fb_velo,xRV,Stuff+
16614,1358089441,FA,J. Cullen,URI,R,341,91.595910,21.455902,10.417795,2257.518696,6.102822,1.108587,6.004221,89.111456,0.073661,119
7944,1203468032,FA,C. Grotyohann,URI,R,173,90.735434,20.985878,8.971219,2186.473280,6.187250,1.877272,5.882359,88.676504,0.078364,117
18671,1487988943,SL,M. Santos,URI,R,198,81.850110,-0.354423,-16.391414,2649.944813,5.025247,2.655238,5.261400,92.562573,0.056433,115
3325,1110853376,SL,J. Sabbath,URI,R,253,78.684421,4.809984,-11.676591,2367.322806,5.682230,1.795760,5.824940,89.853506,0.058242,113
18669,1487988943,FA,M. Santos,URI,R,192,92.588882,16.130212,7.259698,2465.108502,5.323777,2.400913,5.615804,92.562573,0.083881,113
14884,1336247552,FA,C. Maloney,URI,R,96,90.652597,18.868405,9.670062,2206.259268,5.815449,2.210019,5.965383,90.394947,0.087050,112
16615,1358089441,FC,J. Cullen,URI,R,128,82.487807,2.032523,-4.802381,2597.068783,5.840402,1.652846,5.504062,89.111456,0.074373,112
1880,1085415168,CH,E. Maloney,URI,R,114,75.722168,10.028653,9.579532,1451.344744,6.934492,0.068813,4.891990,87.229034,0.074676,112
2955,1101042688,FA,A. Jones,URI,R,143,89.682956,21.046223,7.763579,2156.832315,5.835070,1.054245,5.834861,88.680534,0.090107,110
12730,1299756340,SL,P. Aikens,URI,L,183,72.772005,5.277696,16.742062,2663.800719,4.682106,-2.821260,4.208979,82.998339,0.063305,110


In [74]:
stuff.to_parquet("stuff_26.parquet")
stuff_vs_R.to_parquet("stuff_vsR_26.parquet")
stuff_vs_L.to_parquet("stuff_vsL_26.parquet")

In [75]:
# Converting stuff csv to a leaderboard format, where each pitcher/pitch type is in one row
rows = []
ids = list(stuff["pitcherId"].unique())
pitches = list(stuff["pitchType"].unique())

for i in ids:
    x = stuff[stuff["pitcherId"] == i]
    dx = {}
    dx["Pitcher"] = x["Pitcher"].iloc[0]
    dx["pitcherId"] = i
    dx["team"] = x["team"].iloc[0]
    dx["Hand"] = x["Hand"].iloc[0]
    for pt in list(stuff["pitchType"].unique()):
        if pt in list(x["pitchType"].unique()):
            dx["Stf+ " + pt] = x[x["pitchType"] == pt]["Stuff+"].iloc[0].astype("int64")
            dx[pt + "_ct"] = x[x["pitchType"] == pt]["Count"].iloc[0]
        else:
            dx["Stf+ " + pt] = np.nan
    dx["Total"] = x["Count"].sum()
    rows.append(dx)
table = pd.DataFrame(rows).reset_index(drop=True)
stf_cols = [col for col in table.columns if col.startswith("Stf+")]
table[stf_cols] = table[stf_cols].astype("Int64")

# Calculating overall Stuff+ based on weighted average of each Stuff+ value. More pitch usage = more weight
table["Stuff+"] = 0.0
for p in pitches:
    col = "Stf+ " + p
    ct_col = p + "_ct"
    if col in table.columns and ct_col in table.columns:
        mask = table[col].notna()
        table.loc[mask, "Stuff+"] += (
            table.loc[mask, col] * (table.loc[mask, ct_col] / table.loc[mask, "Total"])
        )
table["Stuff+"] = table["Stuff+"].round().astype(int)
table = table[["Pitcher", "pitcherId", "team", "Total", "Hand", "Stf+ FA", "Stf+ SI", "Stf+ FC", "Stf+ CH", "Stf+ FS", "Stf+ SL", "Stf+ ST", "Stf+ CU", "Stuff+"]]

In [76]:
table[table["team"] == "URI"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SI,Stf+ FC,Stf+ CH,Stf+ FS,Stf+ SL,Stf+ ST,Stf+ CU,Stuff+
3206,M. Santos,1487988943,URI,390,R,113,<NA>,<NA>,<NA>,<NA>,115,<NA>,<NA>,114
2862,J. Cullen,1358089441,URI,726,R,119,<NA>,112,90,<NA>,102,<NA>,104,110
579,J. Sabbath,1110853376,URI,768,R,108,<NA>,<NA>,97,<NA>,113,<NA>,103,108
2568,C. Maloney,1336247552,URI,161,R,112,<NA>,<NA>,<NA>,<NA>,102,<NA>,<NA>,108
1372,C. Grotyohann,1203468032,URI,296,R,117,<NA>,92,<NA>,<NA>,98,<NA>,<NA>,107
513,A. Jones,1101042688,URI,262,R,110,<NA>,104,<NA>,<NA>,98,<NA>,102,106
2201,W. Creed,1299086745,URI,68,L,105,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,105
1229,S. Houchens,1194448803,URI,606,L,106,<NA>,<NA>,103,<NA>,102,<NA>,<NA>,104
2737,B. Perry,1342168540,URI,56,R,103,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,103
2208,P. Aikens,1299756340,URI,355,L,91,<NA>,<NA>,<NA>,<NA>,110,<NA>,<NA>,101


In [158]:
table[table["team"] == "ULL"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SI,Stf+ FC,Stf+ CH,Stf+ FS,Stf+ SL,Stf+ ST,Stf+ CU,Stuff+
2313,C. Brasch,1310239714,ULL,975,R,111,<NA>,<NA>,<NA>,<NA>,116,<NA>,115,113
3932,T. Roman,1710566033,ULL,1352,L,113,<NA>,<NA>,115,<NA>,111,<NA>,109,113
4458,H. Pearson,1891273222,ULL,475,L,115,<NA>,<NA>,<NA>,<NA>,99,<NA>,<NA>,113
4682,B. Wilson,1968029751,ULL,61,L,111,<NA>,<NA>,111,<NA>,<NA>,<NA>,<NA>,111
4686,G. Carter,1969112201,ULL,518,R,104,<NA>,<NA>,<NA>,<NA>,121,<NA>,<NA>,110
3588,T. Papenbrock,1602519712,ULL,614,L,107,107,<NA>,<NA>,<NA>,110,<NA>,<NA>,107
2389,A. Herrmann,1311153664,ULL,1543,L,107,<NA>,99,107,<NA>,105,<NA>,102,106
1036,P. Smith,1170700775,ULL,595,R,101,<NA>,<NA>,99,<NA>,103,<NA>,101,102
685,S. Pruitt,1129332403,ULL,1158,R,99,<NA>,<NA>,96,<NA>,106,<NA>,105,101
3468,C. Alfonso,1572952044,ULL,36,R,101,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,101


In [159]:
table[table["Total"] >= 1000].sort_values("Stuff+", ascending=False).head(25)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SI,Stf+ FC,Stf+ CH,Stf+ FS,Stf+ SL,Stf+ ST,Stf+ CU,Stuff+
781,L. Peterson,1142994176,FLA,1392,R,127,<NA>,127,116,<NA>,123,<NA>,119,123
3306,W. Sanford,1518897568,ORE,1445,R,128,<NA>,<NA>,108,<NA>,109,<NA>,114,122
1800,E. Nachtsheim,1256101932,MCNS,1174,R,126,<NA>,85,114,<NA>,111,<NA>,107,121
2027,M. Edwards,1279382784,USC,1415,L,124,<NA>,<NA>,119,<NA>,118,<NA>,119,121
3928,E. Kleinschmit,1709079320,ORST,1165,L,126,<NA>,106,110,<NA>,117,115,<NA>,120
2364,C. West,1310862081,CONN,1049,L,128,<NA>,<NA>,110,<NA>,111,<NA>,113,120
2220,H. Dietz,1301090561,ARK,1306,L,112,<NA>,131,<NA>,<NA>,118,<NA>,124,120
1786,J. Flora,1254299136,UCSB,1428,R,122,<NA>,117,113,<NA>,124,120,<NA>,120
2015,C. Carlon,1279321088,ASU,1217,L,114,<NA>,<NA>,126,<NA>,122,<NA>,117,119
789,C. Turnquist,1143047168,CP,1165,R,124,<NA>,94,111,<NA>,110,<NA>,107,119


In [160]:
# Converting stuff_vs_R csv to a leaderboard format, where each pitcher/pitch type is in one row
rows = []
ids = list(stuff_vs_R["pitcherId"].unique())
pitches = list(stuff_vs_R["pitchType"].unique())

for i in ids:
    x = stuff_vs_R[stuff_vs_R["pitcherId"] == i]
    dx = {}
    dx["Pitcher"] = x["Pitcher"].iloc[0]
    dx["pitcherId"] = i
    dx["team"] = x["team"].iloc[0]
    dx["Hand"] = x["Hand"].iloc[0]
    for pt in list(stuff_vs_R["pitchType"].unique()):
        if pt in list(x["pitchType"].unique()):
            dx["Stf+ " + pt] = x[x["pitchType"] == pt]["Stuff+"].iloc[0].astype("int64")
            dx[pt + "_ct"] = x[x["pitchType"] == pt]["Count"].iloc[0]
        else:
            dx["Stf+ " + pt] = np.nan
    dx["Total"] = x["Count"].sum()
    rows.append(dx)
table_vs_R = pd.DataFrame(rows).reset_index(drop=True)
stf_cols = [col for col in table_vs_R.columns if col.startswith("Stf+")]
table_vs_R[stf_cols] = table_vs_R[stf_cols].astype("Int64")

# Calculating overall Stuff+ based on weighted average of each Stuff+ value. More pitch usage = more weight
table_vs_R["Stuff+"] = 0.0
for p in pitches:
    col = "Stf+ " + p
    ct_col = p + "_ct"
    if col in table_vs_R.columns and ct_col in table_vs_R.columns:
        mask = table_vs_R[col].notna()
        table_vs_R.loc[mask, "Stuff+"] += (
            table_vs_R.loc[mask, col] * (table_vs_R.loc[mask, ct_col] / table_vs_R.loc[mask, "Total"])
        )
table_vs_R["Stuff+"] = table_vs_R["Stuff+"].round().astype(int)
table_vs_R = table_vs_R[["Pitcher", "pitcherId", "team", "Total", "Hand", "Stf+ FA", "Stf+ SI", "Stf+ FC", "Stf+ CH", "Stf+ FS", "Stf+ SL", "Stf+ ST", "Stf+ CU", "Stuff+"]]

In [161]:
table_vs_R[table_vs_R["team"] == "URI"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SI,Stf+ FC,Stf+ CH,Stf+ FS,Stf+ SL,Stf+ ST,Stf+ CU,Stuff+
2998,M. Santos,1487988943,URI,283,R,116,<NA>,<NA>,<NA>,<NA>,115,<NA>,<NA>,115
548,J. Sabbath,1110853376,URI,529,R,111,<NA>,<NA>,<NA>,<NA>,113,<NA>,104,111
2681,J. Cullen,1358089441,URI,474,R,116,<NA>,110,90,<NA>,102,<NA>,103,109
2066,W. Creed,1299086745,URI,36,L,107,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,107
2415,C. Maloney,1336247552,URI,92,R,111,<NA>,<NA>,<NA>,<NA>,101,<NA>,<NA>,107
1286,C. Grotyohann,1203468032,URI,199,R,116,<NA>,92,<NA>,<NA>,<NA>,<NA>,<NA>,106
485,A. Jones,1101042688,URI,175,R,109,<NA>,<NA>,<NA>,<NA>,97,<NA>,100,105
1149,S. Houchens,1194448803,URI,442,L,105,<NA>,<NA>,102,<NA>,101,<NA>,<NA>,103
743,L. Lavigueur,1143021056,URI,176,R,94,<NA>,<NA>,<NA>,<NA>,104,<NA>,<NA>,100
2571,B. Perry,1342168540,URI,39,R,100,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,100


In [162]:
table_vs_R[table_vs_R["Total"] >= 250].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SI,Stf+ FC,Stf+ CH,Stf+ FS,Stf+ SL,Stf+ ST,Stf+ CU,Stuff+
4054,J. Barberi,1862463014,FLA,326,R,129,<NA>,<NA>,<NA>,<NA>,125,<NA>,<NA>,127
739,L. Peterson,1142994176,FLA,699,R,128,<NA>,127,<NA>,<NA>,122,<NA>,118,125
2149,B. Purcell,1307080960,FSU,368,R,127,<NA>,<NA>,122,<NA>,119,<NA>,<NA>,124
3775,N. Helman,1756665469,KENN,256,R,124,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,124
641,D. Whitney,1128102459,ORST,542,R,123,<NA>,<NA>,116,<NA>,129,125,124,124
2439,C. Linder,1337038062,ASU,495,R,127,<NA>,109,<NA>,<NA>,116,<NA>,<NA>,124
2364,C. Randall,1330736128,UCLA,251,R,123,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,123
1162,A. Troy,1195493376,USC,365,R,126,<NA>,<NA>,<NA>,<NA>,116,<NA>,114,123
1679,J. Flora,1254299136,UCSB,826,R,124,<NA>,118,110,<NA>,125,118,<NA>,122
1567,S. Garewal,1238260224,STAN,382,L,126,<NA>,<NA>,117,<NA>,118,<NA>,<NA>,122


In [163]:
# Converting stuff_vs_L csv to a leaderboard format, where each pitcher/pitch type is in one row
rows = []
ids = list(stuff_vs_L["pitcherId"].unique())
pitches = list(stuff_vs_L["pitchType"].unique())

for i in ids:
    x = stuff_vs_L[stuff_vs_L["pitcherId"] == i]
    dx = {}
    dx["Pitcher"] = x["Pitcher"].iloc[0]
    dx["pitcherId"] = i
    dx["team"] = x["team"].iloc[0]
    dx["Hand"] = x["Hand"].iloc[0]
    for pt in list(stuff_vs_L["pitchType"].unique()):
        if pt in list(x["pitchType"].unique()):
            dx["Stf+ " + pt] = x[x["pitchType"] == pt]["Stuff+"].iloc[0].astype("int64")
            dx[pt + "_ct"] = x[x["pitchType"] == pt]["Count"].iloc[0]
        else:
            dx["Stf+ " + pt] = np.nan
    dx["Total"] = x["Count"].sum()
    rows.append(dx)
table_vs_L = pd.DataFrame(rows).reset_index(drop=True)
stf_cols = [col for col in table_vs_L.columns if col.startswith("Stf+")]
table_vs_L[stf_cols] = table_vs_L[stf_cols].astype("Int64")

# Calculating overall Stuff+ based on weighted average of each Stuff+ value. More pitch usage = more weight
table_vs_L["Stuff+"] = 0.0
for p in pitches:
    col = "Stf+ " + p
    ct_col = p + "_ct"
    if col in table_vs_L.columns and ct_col in table_vs_L.columns:
        mask = table_vs_L[col].notna()
        table_vs_L.loc[mask, "Stuff+"] += (
            table_vs_L.loc[mask, col] * (table_vs_L.loc[mask, ct_col] / table_vs_L.loc[mask, "Total"])
        )
        
table_vs_L["Stuff+"] = table_vs_L["Stuff+"].round().astype(int)
table_vs_L = table_vs_L[["Pitcher", "pitcherId", "team", "Total", "Hand", "Stf+ FA", "Stf+ SI", "Stf+ FC", "Stf+ CH", "Stf+ FS", "Stf+ SL", "Stf+ ST", "Stf+ CU", "Stuff+"]]

In [164]:
table_vs_L[table_vs_L["team"] == "URI"].sort_values("Stuff+", ascending=False)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SI,Stf+ FC,Stf+ CH,Stf+ FS,Stf+ SL,Stf+ ST,Stf+ CU,Stuff+
1126,C. Grotyohann,1203468032,URI,54,R,115,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,115
1819,P. Aikens,1299756340,URI,128,L,100,<NA>,<NA>,<NA>,<NA>,114,<NA>,<NA>,109
2353,J. Cullen,1358089441,URI,215,R,120,<NA>,<NA>,88,<NA>,97,<NA>,<NA>,109
422,A. Jones,1101042688,URI,34,R,107,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,107
2128,C. Maloney,1336247552,URI,69,R,111,<NA>,<NA>,<NA>,<NA>,100,<NA>,<NA>,107
2624,M. Santos,1487988943,URI,107,R,104,<NA>,<NA>,<NA>,<NA>,110,<NA>,<NA>,107
262,E. Maloney,1085415168,URI,130,R,100,<NA>,<NA>,113,<NA>,<NA>,<NA>,<NA>,104
1006,S. Houchens,1194448803,URI,162,L,106,<NA>,<NA>,<NA>,<NA>,102,<NA>,<NA>,104
1814,W. Creed,1299086745,URI,32,L,102,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,102
477,J. Sabbath,1110853376,URI,219,R,98,<NA>,<NA>,96,<NA>,106,<NA>,<NA>,99


In [165]:
table_vs_L[table_vs_L["Total"] >= 250].sort_values("Stuff+", ascending=False).head(10)

,Pitcher,pitcherId,team,Total,Hand,Stf+ FA,Stf+ SI,Stf+ FC,Stf+ CH,Stf+ FS,Stf+ SL,Stf+ ST,Stf+ CU,Stuff+
656,C. Turnquist,1143047168,CP,356,R,126,<NA>,<NA>,111,<NA>,<NA>,<NA>,<NA>,123
1215,D. Sheerin,1213263616,LSU,303,R,125,<NA>,<NA>,<NA>,<NA>,114,<NA>,119,122
649,L. Peterson,1142994176,FLA,676,R,124,<NA>,<NA>,117,<NA>,122,<NA>,118,121
2703,W. Sanford,1518897568,ORE,659,R,125,<NA>,<NA>,108,<NA>,<NA>,<NA>,114,121
1478,E. Nachtsheim,1256101932,MCNS,605,R,124,<NA>,<NA>,115,<NA>,105,<NA>,104,121
2182,D. Gutierrez,1341830144,USD,377,R,121,<NA>,<NA>,117,<NA>,<NA>,<NA>,<NA>,120
1829,H. Dietz,1301090561,ARK,469,L,115,<NA>,127,<NA>,<NA>,117,<NA>,121,120
1180,J. DeCaro,1207928320,UNC,666,R,119,<NA>,<NA>,125,<NA>,114,<NA>,116,120
1668,M. Edwards,1279382784,USC,464,L,122,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,116,120
1003,Z. Peters,1193094400,VCU,298,R,124,<NA>,<NA>,<NA>,<NA>,113,<NA>,111,119


In [167]:
table.to_parquet("stuff_table_26.parquet")
table_vs_R.to_parquet("stuff_table_vsR_26.parquet")
table_vs_L.to_parquet("stuff_table_vsL_26.parquet")